# 05. Detecția obiectelor într-o imagine

Modelul **Tinier-YOLO** recunoaște cele 20 de clase PASCAL VOC: persoană; pasăre, pisică, vacă, câine, cal, oaie; avion, bicicletă, barcă, autobuz, automobil, motocicletă, tren; sticlă, scaun, masă, plantă în ghiveci, canapea și televizor/monitor. Nu are o clasă pentru indicatoare rutiere.

![Topologia Tinier-YOLO](../Tinier-YOLO-topology.svg)

Prima și ultima convoluție rulează pe **CPU ARM**, straturile intermediare pe **FPGA** (ponderi pe 1 bit, activări pe 3 biți), iar decodarea detecțiilor și desenarea dreptunghiurilor pe **CPU cu biblioteca nativă Darknet**. Executați celulele în ordine. Opriți celelalte kerneluri FPGA. Importurile `Xlnk` neutilizate trebuie să fi fost eliminate din pachetul instalat. Nu sunt necesare descărcarea modelului sau reinstalarea pachetului.

**Stocare:** nu se creează directoare de rezultate, PNG-uri, scoruri sau jurnale JSON lângă notebook. Biblioteca nativă folosește fișiere temporare mici în `/dev/shm` (RAM), eliminate după procesare și la finalizarea sesiunii. Istoricul previzualizărilor reduse rămâne în RAM până la repornirea kernelului. Jupyter poate salva imaginile afișate în notebook; pentru eliminare folosiți **Cell → All Output → Clear** înainte de salvare. Imaginile și modelele originale nu sunt modificate.

*Versiune tradusă; codul de inferență este păstrat, cu adaptare minimă pentru găsirea directorului de date. Execuția hardware a acestei copii nu a fost validată separat.*

## 1. Modelul instalat, Darknet și imaginile

Executați această celulă. Ea detectează pachetul instalat și verifică biblioteca `/opt/darknet`, etichetele VOC și resursele grafice **înainte** de inițializarea FPGA. Încărcați imagini suplimentare JPG/PNG/BMP în directorul comun **yoloimages**. Sunt selectate numai fișierele imagine.

Este obligatorie biblioteca Darknet personalizată instalată de acest proiect; un pachet pip independent numit `darknet` nu o poate înlocui.

In [ ]:
import sys
import gc
import random
import time
import ctypes
import tempfile
import os
from pathlib import Path

import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
from IPython.display import display
import qnn
from qnn import TinierYolo, utils

QNN_PARAM_DIR = Path(qnn.__file__).resolve().parent / "params"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "yoloimages").is_dir():
    if (NOTEBOOK_DIR.parent / "yoloimages").is_dir():
        NOTEBOOK_DIR = NOTEBOOK_DIR.parent
    else:
        NOTEBOOK_DIR = Path("/home/xilinx/jupyter_notebooks/qnn")
IMAGE_DIR = NOTEBOOK_DIR / "yoloimages"
DARKNET_DIR = Path("/opt/darknet")
VOC_NAMES = DARKNET_DIR / "data/voc.names"
required_files = [QNN_PARAM_DIR / name for name in (
    "tinier-yolo-layers.json", "tinier-yolo-conv0-W.npy", "tinier-yolo-conv8-W.npy",
    "tinier-yolo-conv0-bias.npy", "tinier-yolo-conv8-bias.npy",
    "tinier-yolo-bwn-3bit-relu-nomaxpool.cfg")]
required_files += [DARKNET_DIR / "python/darknet.py", DARKNET_DIR / "libdarknet.so", VOC_NAMES]
# Darknet loads all printable-character PNGs at each size when drawing labels.
required_files += [DARKNET_DIR / "data/labels" / "{}_{}.png".format(char, size)
                   for size in range(8) for char in range(32, 127)]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files missing (first 12):\n" + "\n".join(missing[:12]))
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError("Image folder missing: " + str(IMAGE_DIR))
image_files = sorted(path for path in IMAGE_DIR.iterdir() if path.is_file()
                     and path.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp"))
if not image_files:
    raise FileNotFoundError("Upload an image into " + str(IMAGE_DIR))
if str(DARKNET_DIR / "python") not in sys.path:
    sys.path.insert(0, str(DARKNET_DIR / "python"))
import darknet as dn

dn.lib.parse_network_cfg.argtypes = [ctypes.c_char_p]
dn.lib.parse_network_cfg.restype = ctypes.c_void_p
dn.lib.forward_region_layer_pointer_nolayer.argtypes = [ctypes.c_void_p, ctypes.POINTER(ctypes.c_float)]
dn.lib.forward_region_layer_pointer_nolayer.restype = None
dn.lib.draw_detection_python.argtypes = [ctypes.c_void_p, ctypes.c_char_p,
    ctypes.c_float, ctypes.c_float, ctypes.c_char_p, ctypes.c_char_p, ctypes.c_char_p, ctypes.c_char_p]
dn.lib.draw_detection_python.restype = None
dn.lib.free_network.argtypes = [ctypes.c_void_p]
dn.lib.free_network.restype = None
print("Parameters:", QNN_PARAM_DIR)
print("Image folder:", IMAGE_DIR, "—", len(image_files), "images")

def memory_preview(value):
    if isinstance(value, Image.Image):
        image = value.copy()
    else:
        with Image.open(str(value)) as opened:
            image = opened.convert("RGB")
    image.thumbnail((480, 360))
    return image

def clear_intermediate_files():
    directory = globals().get("RESULT_DIR")
    if directory is not None and directory.is_dir():
        for path in directory.iterdir():
            if path.is_file():
                path.unlink()



## 2. Încărcarea ponderilor și inițializarea modelului

Așteptați mesajul **Model ready**. Celula încarcă ponderile CPU, inițializează rețeaua de detecție nativă și FPGA-ul și pregătește fișierele temporare în RAM. La reexecutare, eliberează mai întâi modelul anterior. Dacă inițializarea eșuează, eliberează resursele deja obținute înainte de a afișa eroarea.

In [ ]:
def release_model():
    global classifier, net_darknet
    try:
        if globals().get("classifier") is not None:
            classifier.deinit_accelerator()
    finally:
        classifier = None
        if globals().get("net_darknet"):
            dn.lib.free_network(net_darknet)
        net_darknet = None
        if globals().get("_result_workspace") is not None:
            _result_workspace.cleanup()

release_model()
for variable in ("conv0_weights", "conv8_weights", "conv0_bias", "conv8_bias"):
    globals().pop(variable, None)
gc.collect()
print("Loading weights...", flush=True)
conv0_weights = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv0-W.npy"),
                        encoding="latin1", allow_pickle=True).transpose(3, 2, 1, 0)
conv8_weights = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv8-W.npy"),
                        encoding="latin1", allow_pickle=True).transpose(3, 2, 1, 0)
conv0_bias = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv0-bias.npy"),
                     encoding="latin1", allow_pickle=True).reshape(-1, 1)
conv8_bias = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv8-bias.npy"),
                     encoding="latin1", allow_pickle=True).reshape(-1, 1)
classifier = None
net_darknet = None
if globals().get("_result_workspace") is not None:
    _result_workspace.cleanup()
if not Path("/dev/shm").is_dir():
    raise RuntimeError("RAM-backed /dev/shm is required; no SD-card fallback is used.")
_result_workspace = tempfile.TemporaryDirectory(prefix="qnn-05-yolo-", dir="/dev/shm")
RESULT_DIR = Path(_result_workspace.name)
try:
    classifier = TinierYolo()
    net = classifier.load_network()
    out_ch, out_dim, _ = net["conv7"]["output"]
    first_ch, first_dim, _ = net["conv1"]["input"]
    if conv0_weights.shape[0] != first_ch or conv0_bias.shape != (first_ch, 1):
        raise ValueError("First-convolution weights do not match the network.")
    if conv8_weights.shape[:2] != (125, out_ch) or conv8_bias.shape != (125, 1):
        raise ValueError("Expected the supplied 20-class, 5-anchor VOC detection head.")
    conv0_bias = np.broadcast_to(conv0_bias, (first_ch, first_dim * first_dim))
    conv8_bias = np.broadcast_to(conv8_bias, (125, out_dim * out_dim))
    net_darknet = dn.lib.parse_network_cfg(str(
        QNN_PARAM_DIR / "tinier-yolo-bwn-3bit-relu-nomaxpool.cfg").encode())
    if not net_darknet:
        raise RuntimeError("Darknet failed to load its network configuration.")
    print("Initializing FPGA...", flush=True)
    classifier.init_accelerator()
except BaseException:
    release_model()
    raise
# Only temporary native-library files are placed in RAM.
RESULT_INDEX = 0
detection_history = []
print("Model ready. Intermediate files use RAM and are removed after processing.")


## 3. Definirea funcțiilor de procesare și detecție

Executați această celulă o singură dată. Ea definește transformarea *letterbox*, etapele CPU/FPGA și desenarea cu Darknet. Imaginile alocate de Darknet sunt eliberate după copiere. Ordinea axelor și cuantizarea modelului original sunt păstrate. Această celulă nu execută încă detecția.

In [ ]:
def load_detector_image(image_path):
    original = None
    letterboxed = None
    try:
        original = dn.load_image(str(image_path).encode(), 0, 0)
        if not original.data:
            raise RuntimeError("Darknet could not read " + str(image_path))
        letterboxed = dn.letterbox_image(original, 416, 416)
        if not letterboxed.data:
            raise RuntimeError("Darknet letterboxing failed.")
        pixels = np.copy(np.ctypeslib.as_array(letterboxed.data, (3, 416, 416)))
        # Preserve the original trained model's paired spatial-axis swaps.
        return np.swapaxes(pixels, 0, 2)[np.newaxis, :, :, :]
    finally:
        if letterboxed is not None and letterboxed.data:
            dn.free_image(letterboxed)
        if original is not None and original.data:
            dn.free_image(original)

def first_convolution(pixels):
    if pixels.shape != (1, 416, 416, 3):
        raise ValueError("Expected one letterboxed 416x416 RGB image.")
    values = utils.conv_layer(pixels, conv0_weights, b=conv0_bias, stride=2, padding=1)
    return utils.quantize(values.clip(0.0, 4.0) / 4.0, 3)

def fpga_layers(quantized):
    if classifier is None:
        raise RuntimeError("Model released. Rerun section 2 before inference.")
    output = classifier.get_accel_buffer(out_ch, out_dim)
    inputs = np.ascontiguousarray(classifier.prepare_buffer(quantized * 7))
    started = time.perf_counter()
    classifier.inference(inputs, output)
    call_seconds = time.perf_counter() - started
    values = classifier.postprocess_buffer(output)
    if values.size != out_ch * out_dim * out_dim or not np.all(np.isfinite(values)):
        raise ValueError("Invalid FPGA output.")
    return values, call_seconds

def last_convolution(values):
    pixels = np.swapaxes(values.reshape(out_dim, out_dim, out_ch), 0, 1)[np.newaxis]
    output = utils.conv_layer(pixels, conv8_weights, b=conv8_bias, stride=1)
    # The native region layer reads float*, so float32 and contiguous storage are required.
    output = np.ascontiguousarray(output, dtype=np.float32)
    if output.shape != (125, out_dim, out_dim) or not np.all(np.isfinite(output)):
        raise ValueError("Invalid detection-head output.")
    return output

def draw_detections(image_path, values, threshold=0.3):
    global RESULT_INDEX
    if net_darknet is None:
        raise RuntimeError("Model released. Rerun section 2.")
    if not 0 < threshold < 1:
        raise ValueError("Detection threshold must be between 0 and 1.")
    # Defensive check also protects callers that rerun this cell independently.
    if values.dtype != np.float32 or not values.flags.c_contiguous:
        raise ValueError("Darknet input must be contiguous float32.")
    if values.shape != (125, out_dim, out_dim) or not np.all(np.isfinite(values)):
        raise ValueError("Wrong detection-head shape or non-finite values.")
    RESULT_INDEX += 1
    result_name = "detection_{:04d}".format(RESULT_INDEX)
    output_prefix = RESULT_DIR / result_name
    output_image = RESULT_DIR / (result_name + ".png")
    probabilities = RESULT_DIR / (result_name + "_probabilities.txt")
    # This legacy C helper uses 256-byte path buffers, including added suffixes.
    paths = [str(image_path), str(VOC_NAMES), str(DARKNET_DIR) + "/",
             str(output_prefix), str(probabilities)]
    if any(len(path.encode()) > 240 for path in paths):
        raise ValueError("Use a shorter notebook/image path for the legacy Darknet helper.")
    for path in (output_image, probabilities):
        if path.exists():
            path.unlink()
    try:
        dn.lib.forward_region_layer_pointer_nolayer(
            net_darknet, values.ctypes.data_as(ctypes.POINTER(ctypes.c_float)))
        dn.lib.draw_detection_python(net_darknet, paths[0].encode(), ctypes.c_float(threshold),
            ctypes.c_float(0.5), paths[1].encode(), paths[2].encode(),
            paths[3].encode(), paths[4].encode())
        if not output_image.is_file() or not probabilities.is_file():
            raise RuntimeError("Darknet did not create the detection outputs.")
        detections = []
        for line in probabilities.read_text().splitlines():
            if not line.strip():
                continue
            name, score = line.rsplit(":", 1)
            detections.append((name.strip(), float(score.strip().rstrip("%")) / 100.0))
        return memory_preview(output_image), sorted(detections, key=lambda item: item[1], reverse=True)
    finally:
        for path in (output_image, probabilities):
            if path.exists():
                path.unlink()


def show_detections(output_image, detections):
    for name, score in detections:
        print("{:20s} {:6.0%}".format(name, score))
    if not detections:
        print("No objects above the selected threshold. This is a valid result.")
    preview = memory_preview(output_image)
    preview.thumbnail((800, 600))
    display(preview)


## 4. Alegerea și afișarea unei imagini

Păstrați `IMAGE_NAME = None` pentru alegerea aleatorie sau introduceți un nume de fișier din **yoloimages**. Celula afișează imaginea originală și pregătește intrarea detectorului. Pentru altă imagine, reluați această secțiune și **5–9**, păstrând modelul încărcat.

In [ ]:
IMAGE_NAME = None
image_path = random.choice(image_files) if IMAGE_NAME is None else IMAGE_DIR / IMAGE_NAME
if not image_path.is_file():
    raise FileNotFoundError("Image not found: " + str(image_path))
print("Input:", image_path.name)
pixels = load_detector_image(image_path)
with Image.open(str(image_path)) as opened:
    preview = opened.convert("RGB")
preview.thumbnail((640, 480))
display(preview)
stage_seconds = {}

## 5. Prima convoluție pe CPU

Celula calculează prima convoluție în precizie completă și cuantizează activările pentru FPGA. La reexecutare, forma intrării rămâne neschimbată; nu se adaugă o dimensiune suplimentară de batch.

In [ ]:
started = time.perf_counter()
quantized = first_convolution(pixels)
stage_seconds["First convolution + quantization (CPU)"] = time.perf_counter() - started
print("First CPU stage: {:.3f} s".format(stage_seconds["First convolution + quantization (CPU)"]))

## 6. Straturile intermediare cuantizate pe FPGA

Timpul raportat măsoară apelul de inferență prin offload, inclusiv costurile runtime. Pregătirea bufferelor și conversia ieșirii sunt în afara acestui timp.

In [ ]:
fpga_output, offload_seconds = fpga_layers(quantized)
stage_seconds["Quantized-layer offload (FPGA)"] = offload_seconds
print("FPGA offload call: {:.3f} s".format(offload_seconds))

## 7. Ultima convoluție pe CPU

Celula calculează ieșirea stratului de detecție și pregătește valorile `float32` contigue pentru decodorul nativ Darknet.

In [ ]:
started = time.perf_counter()
head_output = last_convolution(fpga_output)
stage_seconds["Last convolution (CPU)"] = time.perf_counter() - started
print("Last CPU convolution: {:.3f} s".format(stage_seconds["Last convolution (CPU)"]))

## 8. Decodarea și afișarea obiectelor detectate

Darknet desenează un **dreptunghi în jurul fiecărui obiect detectat**, afișând clasa și scorul prezis. `THRESHOLD = 0.3` păstrează scorurile peste 30%; un prag mai mare reduce numărul de detecții. Scorurile reprezintă încrederea modelului, nu acuratețea măsurată. Este posibil să nu fie detectat niciun obiect.

Imaginea și scorurile provin din aceeași inferență CPU/FPGA, iar rezultatele sunt ordonate numeric. Previzualizările sunt păstrate exclusiv în RAM. Rezultatele se acumulează mai jos; reexecutarea secțiunii afișează istoricul complet al sesiunii curente. Reexecutarea secțiunii 2 începe o sesiune nouă.

In [ ]:
THRESHOLD = 0.3
started = time.perf_counter()
output_image, detections = draw_detections(image_path, head_output, THRESHOLD)
stage_seconds["Decode + draw + file I/O (CPU)"] = time.perf_counter() - started
detection_history.append(dict(image=image_path.name, output=output_image,
                              detections=list(detections)))
# Jupyter clears a cell when it is rerun: render the complete session history.
for number, result in enumerate(detection_history, 1):
    print("\nResult {} — {}".format(number, result["image"]))
    show_detections(result["output"], result["detections"])
    print("Saved:", result["output"])

## 9. Analiza timpilor măsurați pe etape

Axa orizontală reprezintă **secunde**. Barele indică etape diferite ale aceleiași inferențe, nu accelerarea aceleiași operații de la CPU la FPGA. Suma exclude citirea inițială a imaginii, conversia bufferelor, afișarea în notebook și pauzele dintre celule; nu reprezintă FPS al camerei.

In [ ]:
stage_order = ["First convolution + quantization (CPU)", "Quantized-layer offload (FPGA)",
               "Last convolution (CPU)", "Decode + draw + file I/O (CPU)"]
if any(name not in stage_seconds for name in stage_order):
    raise RuntimeError("Run sections 5–8 for the current image first.")
values = [stage_seconds[name] for name in stage_order]
for name, seconds in zip(stage_order, values):
    print("{}: {:.3f} s".format(name, seconds))
print("Sum of measured stages: {:.3f} s".format(sum(values)))
fig, ax = plt.subplots(figsize=(9, 3.5))
positions = np.arange(len(stage_order))
ax.barh(positions, values, color=["steelblue", "darkorange", "steelblue", "steelblue"])
ax.set_yticks(positions)
ax.set_yticklabels(stage_order)
ax.invert_yaxis()
ax.set_xlabel("Measured stage time (seconds)")
ax.set_title("Tinier-YOLO inference stages")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)

## 10. Finalizarea

Executați celula pentru a elibera runtime-ul FPGA și rețeaua Darknet. Nu se salvează PNG-uri cu rezultate. Pentru reluare, executați secțiunea 2 și apoi 4–9. Închideți kernelul înainte de alt notebook FPGA.

In [ ]:
release_model()
for variable in ("conv0_weights", "conv8_weights", "conv0_bias", "conv8_bias",
                 "pixels", "quantized", "fpga_output", "head_output"):
    globals().pop(variable, None)
gc.collect()
print("Accelerator and Darknet network released; large arrays cleared.")

**Validare:** fluxul Python și tratarea erorilor au fost verificate off-board cu interfețe simulate și operațiile NumPy originale. Execuția pe FPGA, biblioteca Darknet nativă și calitatea predicțiilor necesită în continuare verificare pe placa PYNQ.